<a href="https://colab.research.google.com/github/danielbr9382/Corpus2GeoRAG_Pacific/blob/main/NER_COMBINED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

תא 1 — התקנות (Python)

In [1]:
%pip install -q "transformers==4.49.0" "spacy>=3.8,<3.9" "spacy-transformers>=1.3,<1.4" "gliner2[local]==2.1.0" "flair==0.15.1" "huggingface-hub>=0.30,<1" "packaging>=24,<27"
!python -m spacy download en_core_web_trf
!python -m pip check

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 27.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.8/355.8 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.6/779.6 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

תא 2 — כל הייבואים (Python)

In [2]:
import gc
import hashlib
import json
import logging
import platform
import random
import re
import subprocess
import time
import uuid
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import spacy
import torch
import transformers
from flair.models import SequenceTagger
from flair.splitter import SegtokSentenceSplitter
from gliner2 import AutoExtractor
from google.colab import files
from huggingface_hub import hf_hub_download, model_info, snapshot_download
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("colab_ner")
print ("כל הספריות נטענו")

כל הספריות נטענו


תא 3 — הגדרות מרכזיות (Python)

In [3]:
SAMPLE_SIZE = 300
SEED = 17

# הנתיב המקורי מהמחברת. אפשר להצביע במקום זאת ישירות על DATA.json קיים ב-Colab.
REPO_URL = "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git"
REPO_DIR = Path("/content/Corpus2GeoRAG_Pacific")
SOURCE_PATH = None  # למשל Path("/content/DATA.json")

# נתיבים לקובצי sample_*.json קודמים, אם רוצים לדווח חפיפה.
PREVIOUS_SAMPLE_PATHS = []

HF_STRIDE = 64
GLINER_THRESHOLD = 0.5
GLINER_CHUNK_SIZE = 384
GLINER_CHUNK_OVERLAP = 64
HIGH_SINGLETON_SCORE = 0.99

BERT_ID = "dslim/bert-base-NER"
BERT_REVISION = "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"
ROBERTA_ID = "Jean-Baptiste/roberta-large-ner-english"
SPACY_ID = "en_core_web_trf"
GLINER_ID = "fastino/gliner2.5-base-v1"
FLAIR_ID = "flair/ner-english-ontonotes-large"

GLINER_LABELS = [
    "person", "organization", "military unit", "location",
    "geopolitical entity", "facility", "nationality or political group",
    "event", "date", "time", "weapon", "vehicle", "product",
    "work of art", "law", "language", "money", "quantity",
    "percent", "ordinal", "cardinal", "miscellaneous named entity",
]

assert SAMPLE_SIZE > 0 and isinstance(SEED, int)
assert len(GLINER_LABELS) == 22
assert 0 <= GLINER_CHUNK_OVERLAP < GLINER_CHUNK_SIZE

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = Path("/content/TMP_NER") / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
ZIP_PATH = Path("/content") / f"ner_run_{RUN_ID}.zip"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()

def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())

# def save_json(path, obj):
#     with Path(path).open("x", encoding="utf-8") as handle:
#         json.dump(obj, handle, ensure_ascii=False, indent=2)
def save_json(path, obj):
    payload = json.dumps(
        obj,
        ensure_ascii=False,
        indent=2,
        default=jsonable,
        allow_nan=False,
    )
    with Path(path).open("x", encoding="utf-8") as handle:
        handle.write(payload)


def jsonable(value):
    if hasattr(value, "item"):
        return value.item()
    raise TypeError(f"Cannot serialize {type(value).__name__}")

def package_versions():
    names = ["torch", "transformers", "spacy", "spacy-transformers",
             "gliner2", "flair", "huggingface-hub"]
    return {name: version(name) for name in names}

PACKAGE_VERSIONS = package_versions()
MODEL_FILES = {}
print("RUN_ID:", RUN_ID)
print("ספריות:", PACKAGE_VERSIONS)
print("GPU:", torch.cuda.is_available())

RUN_ID: 20261011T190026Z_ce33d695
ספריות: {'torch': '2.11.0+cu130', 'transformers': '4.49.0', 'spacy': '3.8.16', 'spacy-transformers': '1.3.9', 'gliner2': '2.1.0', 'flair': '0.15.1', 'huggingface-hub': '0.36.2'}
GPU: True


תא 4 — טעינת המקור (Python)

In [4]:
if SOURCE_PATH is None:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)],
            check=True,
        )
    candidates = sorted((REPO_DIR / "Data").glob("*.json"))
    if len(candidates) != 1:
        raise ValueError(
            f"נדרש קובץ מקור JSON יחיד בתיקיית Data; נמצאו: {[p.name for p in candidates]}. "
            "הגדר SOURCE_PATH במפורש."
        )
    input_path = candidates[0]
else:
    input_path = Path(SOURCE_PATH)

if not input_path.is_file():
    raise FileNotFoundError(input_path)

source_bytes = input_path.read_bytes()
source_sha256 = sha256_bytes(source_bytes)
data = json.loads(source_bytes)

# זהו מבנה המקור שבו משתמשת המחברת הקיימת: כותרת מאמר -> טקסט מלא.
if not isinstance(data, dict) or not data:
    raise ValueError("קובץ המקור חייב להיות אובייקט JSON לא ריק")
if not all(isinstance(title, str) and isinstance(text, str)
           for title, text in data.items()):
    raise ValueError("צפוי מבנה JSON של article_title -> text; אין להמציא מזהי מקור")

print("מקור:", input_path.name)
print("מאמרים:", len(data))
print("SHA256 מקור:", source_sha256)

מקור: DATA.json
מאמרים: 428
SHA256 מקור: d58044161d4ec56f260852b9a773a9858b09438a97c00d8477c5dcc1101d71a2


תא 5 — חילוץ פסקאות ומדגם יחיד (Python)

In [5]:
paragraphs = []

for title, text in data.items():
    # כלל חילוץ הפסקאות וההיסטים מהמחברת המקורית.
    separators = list(re.finditer(r"\n[ \t]*\n+", text))
    starts = [0] + [match.end() for match in separators]
    ends = [match.start() for match in separators] + [len(text)]

    for index, (left, right) in enumerate(zip(starts, ends)):
        chunk = text[left:right]
        stripped = chunk.strip()
        if not stripped or re.fullmatch(r"=+\s*.*?\s*=+", stripped):
            continue

        start = left + len(chunk) - len(chunk.lstrip())
        end = left + len(chunk.rstrip())
        paragraphs.append({
            "article_title": title,
            "paragraph_index": index,
            "start": start,
            "end": end,
            "text": text[start:end],
        })

paragraphs.sort(
    key=lambda p: (p["article_title"], p["start"], p["paragraph_index"])
)
if SAMPLE_SIZE > len(paragraphs):
    raise ValueError(
        f"ביקשת {SAMPLE_SIZE} פסקאות; זמינות רק {len(paragraphs)}"
    )

sample = random.Random(SEED).sample(paragraphs, SAMPLE_SIZE)
sample_path = RUN_DIR / f"sample_{SAMPLE_SIZE}_paragraphs_seed{SEED}.json"

sample_data = {
    "schema_version": "ner-experiment-1",
    "run_id": RUN_ID,
    "source_file": input_path.name,
    "source_sha256": source_sha256,
    "seed": SEED,
    "sample_size": SAMPLE_SIZE,
    "sampling_method": "sorted_paragraphs_random.Random(SEED).sample",
    "paragraphs": sample,
}
save_json(sample_path, sample_data)
sample_sha256 = sha256_file(sample_path)

def paragraph_key(paragraph):
    return (
        paragraph["article_title"],
        paragraph["paragraph_index"],
        paragraph["start"],
        paragraph["end"],
    )

current_by_key = {
    paragraph_key(paragraph): {
        "sample_index": index,
        "article_title": paragraph["article_title"],
        "paragraph_index": paragraph["paragraph_index"],
        "start": paragraph["start"],
        "end": paragraph["end"],
    }
    for index, paragraph in enumerate(sample)
}

# קבצים שהוגדרו במפורש, וכן קובצי מדגם/ZIP קודמים הזמינים ב-Colab.
explicit_paths = [Path(path) for path in PREVIOUS_SAMPLE_PATHS]
for path in explicit_paths:
    if not path.is_file():
        raise FileNotFoundError(f"קובץ מדגם קודם שהוגדר אינו זמין: {path}")

discovered_paths = set(explicit_paths)
discovered_paths.update(Path("/content").glob("sample_*.json"))
discovered_paths.update(Path("/content").glob("ner_run_*.zip"))
discovered_paths.update(Path("/content/TMP_NER").glob("**/sample_*.json"))
discovered_paths.update((REPO_DIR / "Data" / "TMP_NER").glob("**/sample_*.json"))
discovered_paths.discard(sample_path)

overlap_files = []
overlap_union = set()
comparable_count = 0

def compare_previous_sample(label, previous):
    global comparable_count
    previous_paragraphs = previous.get("paragraphs")
    if not isinstance(previous_paragraphs, list):
        raise ValueError(f"קובץ מדגם קודם אינו מכיל paragraphs: {label}")

    previous_source_hash = previous.get("source_sha256")
    if previous_source_hash and previous_source_hash != source_sha256:
        overlap_files.append({
            "file": label,
            "status": "source_mismatch",
            "overlap_count": None,
            "overlapping_paragraphs": [],
        })
        return

    previous_keys = {paragraph_key(p) for p in previous_paragraphs}
    shared_keys = current_by_key.keys() & previous_keys
    comparable_count += 1
    overlap_union.update(shared_keys)
    overlap_files.append({
        "file": label,
        "status": "checked",
        "source_hash_check": (
            "matched" if previous_source_hash else "unavailable"
        ),
        "overlap_count": len(shared_keys),
        "overlapping_paragraphs": [
            current_by_key[key] for key in sorted(shared_keys)
        ],
    })

for old_path in sorted(discovered_paths, key=str):
    if old_path.resolve() == sample_path.resolve():
        continue

    if old_path.suffix.lower() == ".json":
        previous = json.loads(old_path.read_text(encoding="utf-8"))
        compare_previous_sample(str(old_path), previous)

    elif old_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(old_path) as archive:
            sample_members = sorted(
                name for name in archive.namelist()
                if name.startswith("TMP_NER/sample_") and name.endswith(".json")
            )
            if not sample_members:
                overlap_files.append({
                    "file": str(old_path),
                    "status": "no_sample_file",
                    "overlap_count": None,
                    "overlapping_paragraphs": [],
                })
            for member in sample_members:
                previous = json.loads(archive.read(member))
                compare_previous_sample(f"{old_path}!{member}", previous)

overlap_report = {
    "status": "checked" if comparable_count else "not_checked",
    "comparable_files": comparable_count,
    "overlap_count": len(overlap_union) if comparable_count else None,
    "overlapping_paragraphs": [
        current_by_key[key] for key in sorted(overlap_union)
    ],
    "files": overlap_files,
}

print(f"מדגם: {SAMPLE_SIZE} מתוך {len(paragraphs)} פסקאות; SEED={SEED}")
print("קובץ:", sample_path)
print("SHA256 מדגם:", sample_sha256)
print("בדיקת חפיפה:", overlap_report["status"])

if overlap_report["status"] == "checked":
    print("מספר פסקאות חופפות ייחודיות:", overlap_report["overlap_count"])
    print("מזהי הפסקאות החופפות:")
    for item in overlap_report["overlapping_paragraphs"]:
        print(item)
else:
    print("not_checked — לא נמצא קובץ מדגם קודם בר־השוואה")

for item in overlap_report["files"]:
    print(
        "קובץ קודם:", item["file"],
        "| מצב:", item["status"],
        "| חפיפה:", item["overlap_count"],
    )

# paragraphs = []

# for title, text in data.items():
#     # אותו כלל הפרדה ואותה התאמת היסטים מהמחברת המצורפת.
#     separators = list(re.finditer(r"\n[ \t]*\n+", text))
#     starts = [0] + [match.end() for match in separators]
#     ends = [match.start() for match in separators] + [len(text)]

#     for index, (left, right) in enumerate(zip(starts, ends)):
#         chunk = text[left:right]
#         stripped = chunk.strip()
#         if not stripped or re.fullmatch(r"=+\s*.*?\s*=+", stripped):
#             continue

#         start = left + len(chunk) - len(chunk.lstrip())
#         end = left + len(chunk.rstrip())
#         paragraphs.append({
#             "article_title": title,
#             "paragraph_index": index,
#             "start": start,
#             "end": end,
#             "text": text[start:end],
#         })

# paragraphs.sort(key=lambda p: (p["article_title"], p["start"], p["paragraph_index"]))
# if SAMPLE_SIZE > len(paragraphs):
#     raise ValueError(f"ביקשת {SAMPLE_SIZE} פסקאות; זמינות רק {len(paragraphs)}")

# sample = random.Random(SEED).sample(paragraphs, SAMPLE_SIZE)
# sample_path = RUN_DIR / f"sample_{SAMPLE_SIZE}_paragraphs_seed{SEED}.json"

# sample_data = {
#     "schema_version": "ner-experiment-1",
#     "run_id": RUN_ID,
#     "source_file": input_path.name,
#     "source_sha256": source_sha256,
#     "seed": SEED,
#     "sample_size": SAMPLE_SIZE,
#     "sampling_method": "sorted_paragraphs_random.Random(SEED).sample",
#     "paragraphs": sample,
# }
# # DATA.json המקורי אינו מספק revision_id לכל מאמר, ולכן אין שדה מומצא כזה.
# save_json(sample_path, sample_data)
# sample_sha256 = sha256_file(sample_path)

# def paragraph_key(p):
#     return (p["article_title"], p["paragraph_index"], p["start"], p["end"])

# current_keys = {paragraph_key(p) for p in sample}
# overlap_report = {}
# for old_path in PREVIOUS_SAMPLE_PATHS:
#     old_path = Path(old_path)
#     old_data = json.loads(old_path.read_text(encoding="utf-8"))
#     old_keys = {paragraph_key(p) for p in old_data["paragraphs"]}
#     overlap_report[str(old_path)] = len(current_keys & old_keys)

# print(f"מדגם: {SAMPLE_SIZE} מתוך {len(paragraphs)} פסקאות; SEED={SEED}")
# print("קובץ:", sample_path)
# print("SHA256 מדגם:", sample_sha256)
# print("חפיפה עם מדגמים קודמים:", overlap_report)

מדגם: 300 מתוך 9496 פסקאות; SEED=17
קובץ: /content/TMP_NER/20261011T190026Z_ce33d695/sample_300_paragraphs_seed17.json
SHA256 מדגם: 303d00704f8e4bcbe2bcf56e0c4631c75eb64aaeef9ffbcae8d43be2a2fd1bec
בדיקת חפיפה: checked
מספר פסקאות חופפות ייחודיות: 4
מזהי הפסקאות החופפות:
{'sample_index': 123, 'article_title': 'British Empire in World War II', 'paragraph_index': 29, 'start': 27060, 'end': 27448}
{'sample_index': 136, 'article_title': 'Declaration by United Nations', 'paragraph_index': 11, 'start': 7129, 'end': 7378}
{'sample_index': 137, 'article_title': 'Diplomatic history of World War II', 'paragraph_index': 48, 'start': 32595, 'end': 34427}
{'sample_index': 174, 'article_title': 'Supreme Court of Judicature of Japan', 'paragraph_index': 0, 'start': 0, 'end': 1233}
קובץ קודם: /content/Corpus2GeoRAG_Pacific/Data/TMP_NER/sample_100_paragraphs_seed42.json | מצב: checked | חפיפה: 4


תא 6 — בדיקות מדגם וכלי שמירה משותפים (Python)

In [6]:
def validate_sample(sample_rows, source_data):
    if len(sample_rows) != SAMPLE_SIZE:
        raise ValueError(f"ציפינו ל-{SAMPLE_SIZE} פסקאות, התקבלו {len(sample_rows)}")
    if len({paragraph_key(p) for p in sample_rows}) != SAMPLE_SIZE:
        raise ValueError("פסקה מופיעה במדגם יותר מפעם אחת")
    for index, p in enumerate(sample_rows):
        title, start, end = p["article_title"], p["start"], p["end"]
        if title not in source_data or not 0 <= start < end <= len(source_data[title]):
            raise ValueError(f"היסט מסמך לא תקין בפסקה {index}")
        if source_data[title][start:end] != p["text"]:
            raise ValueError(f"טקסט פסקה {index} שונה מחיתוך המקור")
        if end - start != len(p["text"]):
            raise ValueError(f"אורך פסקה {index} אינו תואם להיסטים")

def make_mention(paragraph, label, model_text, start, end, score,
                 raw_reference, extra=None):
    mention = {
        "text": model_text,
        "label": label,
        "start": start,
        "end": end,
        "start_document": None,
        "end_document": None,
        "score": float(score) if score is not None else None,
        "kind_score": "model_raw" if score is not None else None,
        "ref_output_raw": raw_reference,
        "status_validation": "invalid",
        "error": None,
    }
    if extra:
        mention.update(extra)

    text = paragraph["text"]
    valid = (
        isinstance(start, int) and isinstance(end, int)
        and 0 <= start < end <= len(text)
        and text[start:end] == model_text
    )
    if valid:
        mention["start_document"] = paragraph["start"] + start
        mention["end_document"] = paragraph["start"] + end
        mention["status_validation"] = "valid"
    else:
        mention["error"] = "offsets or model text do not match original paragraph"
    return mention

def check_mention(paragraph, mention):
    if mention["status_validation"] != "valid":
        return
    start, end = mention["start"], mention["end"]
    text = paragraph["text"]
    assert 0 <= start < end <= len(text)
    assert text[start:end] == mention["text"]
    assert mention["start_document"] == paragraph["start"] + start
    assert mention["end_document"] == paragraph["start"] + end
    assert data[paragraph["article_title"]][
        mention["start_document"]:mention["end_document"]
    ] == mention["text"]

def run_model(short_name, model_id, revision_model, revision_tokenizer,
              labels, settings, extractor):
    results = []
    for index, paragraph in enumerate(sample):
        started = time.perf_counter()
        try:
            mentions, raw = extractor(paragraph)
            for mention in mentions:
                check_mention(paragraph, mention)
            invalid = [m for m in mentions if m["status_validation"] == "invalid"]
            status = "partial_invalid" if invalid else "success"
            error = "; ".join(m["error"] or "invalid mention" for m in invalid) or None
        except Exception as exc:
            log.exception("%s failed on sample_index=%d", short_name, index)
            mentions, raw = [], None
            status = "error"
            error = f"{type(exc).__name__}: {exc}"

        results.append({
            "sample_index": index,
            "paragraph": paragraph,
            "mentions": mentions,
            "pipeline_output": raw,
            "raw_output": raw,
            "status_processing": status,
            "error": error,
            "processing_seconds": round(time.perf_counter() - started, 4),
            "at_processed": utc_now(),
        })
        if (index + 1) % 10 == 0 or index + 1 == SAMPLE_SIZE:
            print(f"{short_name}: {index + 1}/{SAMPLE_SIZE}")

    output = {
        "schema_version": "ner-experiment-1",
        "run_id": RUN_ID,
        "model_id": model_id,
        "name_model": model_id,
        "revision_model": revision_model,
        "revision_tokenizer": revision_tokenizer,
        "library_versions": PACKAGE_VERSIONS,
        "python_version": platform.python_version(),
        "sample_file": sample_path.name,
        "sample_sha256": sample_sha256,
        "source_file": input_path.name,
        "source_sha256": source_sha256,
        "seed": SEED,
        "sample_size": SAMPLE_SIZE,
        "labels": labels,
        "settings": settings,
        "configuration_sha256": sha256_bytes(
            json.dumps({"model": model_id, "revision": revision_model,
                        "labels": labels, "settings": settings},
                       sort_keys=True).encode("utf-8")
        ),
        "at_processed": utc_now(),
        "results": results,
    }
    path = RUN_DIR / f"ner_{short_name}.json"
    save_json(path, output)
    MODEL_FILES[short_name] = path
    counts = {state: sum(r["status_processing"] == state for r in results)
              for state in ("success", "partial_invalid", "error")}
    print(short_name, counts, "נשמר:", path.name)
    return output

def release_model(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

validate_sample(sample, data)
assert sample_sha256 == sha256_file(sample_path)
print(f"כל {SAMPLE_SIZE} הפסקאות תואמות לטקסט המקור ולהיסטים במסמך")

כל 300 הפסקאות תואמות לטקסט המקור ולהיסטים במסמך


תא 7 — כלי Hugging Face משותף (Python)

In [7]:
def load_hf_ner(model_id, revision):
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, revision=revision, use_fast=True
    )
    if not tokenizer.is_fast:
        raise RuntimeError(f"{model_id} did not load a fast tokenizer")
    model = AutoModelForTokenClassification.from_pretrained(
        model_id, revision=revision
    )
    ner = pipeline(
        "ner",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="average",
        stride=HF_STRIDE,
        device=0 if torch.cuda.is_available() else -1,
    )
    return tokenizer, model, ner

def extract_hf(paragraph, ner):
    predictions = ner(paragraph["text"])
    mentions = []
    for ordinal, prediction in enumerate(predictions):
        mentions.append(make_mention(
            paragraph=paragraph,
            label=prediction["entity_group"],
            model_text=paragraph["text"][
                int(prediction["start"]):int(prediction["end"])
            ] if 0 <= int(prediction["start"]) < int(prediction["end"]) <= len(paragraph["text"])
              else prediction.get("word"),
            start=int(prediction["start"]),
            end=int(prediction["end"]),
            score=prediction.get("score"),
            raw_reference=f"results[].pipeline_output[{ordinal}]",
            extra={"model_word": prediction.get("word")},
        ))
    return mentions, predictions

# בדיקת חוזה: pipeline מחזיר היסטים בפסקה; אין כאן חלונות ידניים.
# stride ו-fast tokenizer משמשים את pipeline לחלונות חופפים.

תא 8 — BERT (Python)

In [8]:
bert_tokenizer, bert_model, bert_ner = load_hf_ner(BERT_ID, BERT_REVISION)
bert_revision_tokenizer = getattr(bert_tokenizer, "_commit_hash", None) or BERT_REVISION
bert_labels = sorted(set(bert_model.config.id2label.values()))

bert_output = run_model(
    "bert", BERT_ID, BERT_REVISION, bert_revision_tokenizer,
    bert_labels,
    {"aggregation_strategy": "average", "stride_tokens": HF_STRIDE,
     "fast_tokenizer": True, "threshold": None},
    lambda paragraph: extract_hf(paragraph, bert_ner),
)
del bert_ner, bert_model, bert_tokenizer
release_model()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


bert: 10/300
bert: 20/300
bert: 30/300
bert: 40/300
bert: 50/300
bert: 60/300
bert: 70/300
bert: 80/300
bert: 90/300
bert: 100/300
bert: 110/300
bert: 120/300
bert: 130/300
bert: 140/300
bert: 150/300
bert: 160/300
bert: 170/300
bert: 180/300
bert: 190/300
bert: 200/300
bert: 210/300
bert: 220/300
bert: 230/300
bert: 240/300
bert: 250/300
bert: 260/300
bert: 270/300
bert: 280/300
bert: 290/300
bert: 300/300
bert {'success': 300, 'partial_invalid': 0, 'error': 0} נשמר: ner_bert.json


תיקון לשגיאה

In [9]:
# import sys

# # התיקון ישמש גם את ארבעת המודלים הבאים.
# def save_json(path, obj):
#     with Path(path).open("x", encoding="utf-8") as handle:
#         json.dump(obj, handle, ensure_ascii=False, indent=2, default=jsonable)

# # ניסיון להציל את תוצאות BERT שכבר חושבו מתוך מסגרת השגיאה.
# recovered_output = None
# traceback = getattr(sys, "last_traceback", None)
# while traceback is not None:
#     frame = traceback.tb_frame
#     if frame.f_code.co_name == "run_model":
#         recovered_output = frame.f_locals.get("output")
#     traceback = traceback.tb_next

# if recovered_output is not None:
#     recovered_path = RUN_DIR / f"ner_bert_recovered_{uuid.uuid4().hex[:8]}.json"
#     save_json(recovered_path, recovered_output)
#     bert_output = recovered_output
#     MODEL_FILES["bert"] = recovered_path
#     print("תוצאות BERT נשמרו בלי להריץ את המודל שוב:", recovered_path)
# else:
#     print("לא ניתן לשחזר את התוצאות מזיכרון השגיאה; יהיה צורך להריץ שוב את BERT.")

תא 9 — RoBERTa (Python)

In [10]:
roberta_revision = model_info(ROBERTA_ID).sha
roberta_tokenizer, roberta_model, roberta_ner = load_hf_ner(
    ROBERTA_ID, roberta_revision
)
roberta_revision_tokenizer = (
    getattr(roberta_tokenizer, "_commit_hash", None) or roberta_revision
)
roberta_labels = sorted(set(roberta_model.config.id2label.values()))

roberta_output = run_model(
    "roberta", ROBERTA_ID, roberta_revision, roberta_revision_tokenizer,
    roberta_labels,
    {"aggregation_strategy": "average", "stride_tokens": HF_STRIDE,
     "fast_tokenizer": True, "threshold": None},
    lambda paragraph: extract_hf(paragraph, roberta_ner),
)
del roberta_ner, roberta_model, roberta_tokenizer
release_model()

tokenizer_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/849 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.13/dist-packages/transformers/pipelines/token_classification.py:398: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


roberta: 10/300
roberta: 20/300
roberta: 30/300
roberta: 40/300
roberta: 50/300
roberta: 60/300
roberta: 70/300
roberta: 80/300
roberta: 90/300
roberta: 100/300
roberta: 110/300
roberta: 120/300
roberta: 130/300
roberta: 140/300
roberta: 150/300
roberta: 160/300
roberta: 170/300
roberta: 180/300
roberta: 190/300
roberta: 200/300
roberta: 210/300
roberta: 220/300
roberta: 230/300
roberta: 240/300
roberta: 250/300
roberta: 260/300
roberta: 270/300
roberta: 280/300
roberta: 290/300
roberta: 300/300
roberta {'success': 300, 'partial_invalid': 0, 'error': 0} נשמר: ner_roberta.json


תא 10 — spaCy en_core_web_trf (Python)

In [11]:
spacy_nlp = spacy.load(
    SPACY_ID,
    disable=["tagger", "parser", "attribute_ruler", "lemmatizer"],
)
spacy_labels = sorted(spacy_nlp.get_pipe("ner").labels)

def extract_spacy(paragraph):
    doc = spacy_nlp(paragraph["text"])
    raw = [
        {"text": entity.text, "label": entity.label_,
         "start": entity.start_char, "end": entity.end_char}
        for entity in doc.ents
    ]
    mentions = [
        make_mention(
            paragraph, item["label"], item["text"], item["start"], item["end"],
            None, f"results[].pipeline_output[{ordinal}]"
        )
        for ordinal, item in enumerate(raw)
    ]
    return mentions, raw

spacy_output = run_model(
    "spacy", SPACY_ID, spacy_nlp.meta.get("version"), None,
    spacy_labels,
    {"disabled_components": ["tagger", "parser", "attribute_ruler", "lemmatizer"],
     "threshold": None},
    extract_spacy,
)
del spacy_nlp
release_model()

spacy: 10/300
spacy: 20/300
spacy: 30/300
spacy: 40/300
spacy: 50/300
spacy: 60/300
spacy: 70/300
spacy: 80/300
spacy: 90/300
spacy: 100/300
spacy: 110/300
spacy: 120/300
spacy: 130/300
spacy: 140/300
spacy: 150/300
spacy: 160/300
spacy: 170/300
spacy: 180/300
spacy: 190/300
spacy: 200/300
spacy: 210/300
spacy: 220/300
spacy: 230/300
spacy: 240/300
spacy: 250/300
spacy: 260/300
spacy: 270/300
spacy: 280/300
spacy: 290/300
spacy: 300/300
spacy {'success': 300, 'partial_invalid': 0, 'error': 0} נשמר: ner_spacy.json


תא 11 — GLiNER2.5 (Python)

In [12]:
gliner_revision = model_info(GLINER_ID).sha
gliner_dir = snapshot_download(GLINER_ID, revision=gliner_revision)
gliner_model = AutoExtractor.from_pretrained(gliner_dir)

def extract_gliner(paragraph):
    text = paragraph["text"]
    raw = gliner_model.extract_entities_long(
        text,
        GLINER_LABELS,
        chunk_size=GLINER_CHUNK_SIZE,
        chunk_overlap=GLINER_CHUNK_OVERLAP,
        threshold=GLINER_THRESHOLD,
        include_spans=True,
        include_confidence=True,
        overlap_policy="allow",
    )
    mentions = []
    for label, entities in raw["entities"].items():
        for ordinal, entity in enumerate(entities):
            mentions.append(make_mention(
                paragraph, label, entity["text"],
                int(entity["start"]), int(entity["end"]),
                entity.get("confidence"),
                f"results[].pipeline_output.entities[{label!r}][{ordinal}]",
            ))
    return mentions, raw

gliner_output = run_model(
    "gliner", GLINER_ID, gliner_revision, gliner_revision,
    GLINER_LABELS,
    {"threshold": GLINER_THRESHOLD,
     "chunk_size_words": GLINER_CHUNK_SIZE,
     "chunk_overlap_words": GLINER_CHUNK_OVERLAP,
     "overlap_policy": "allow",
     "offset_scope": "paragraph_global"},
    extract_gliner,
)
del gliner_model
release_model()

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

SKILL.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/774M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

You are using a model of type gliner2 to instantiate a model of type extractor. This is not supported for all configurations of models and can yield errors.
/usr/local/lib/python3.13/dist-packages/gliner2/models/boundary/model.py:2212: UserWarning: Checkpoint uses legacy list-valued extra_special_tokens metadata; loading with compatibility normalization.
  tokenizer = load_extractor_tokenizer(tokenizer_source)


gliner: 10/300
gliner: 20/300
gliner: 30/300
gliner: 40/300
gliner: 50/300
gliner: 60/300
gliner: 70/300
gliner: 80/300
gliner: 90/300
gliner: 100/300
gliner: 110/300
gliner: 120/300
gliner: 130/300
gliner: 140/300
gliner: 150/300
gliner: 160/300
gliner: 170/300
gliner: 180/300
gliner: 190/300
gliner: 200/300
gliner: 210/300
gliner: 220/300
gliner: 230/300
gliner: 240/300
gliner: 250/300
gliner: 260/300
gliner: 270/300
gliner: 280/300
gliner: 290/300
gliner: 300/300
gliner {'success': 300, 'partial_invalid': 0, 'error': 0} נשמר: ner_gliner.json


תא 12 — תיקון Flair ובדיקות מקרי קצה (Python)

In [13]:
def _collapse_whitespace_with_map(text):
    normalized, source_starts, source_ends = [], [], []
    position = 0
    while position < len(text):
        if text[position].isspace():
            end = position + 1
            while end < len(text) and text[end].isspace():
                end += 1
            normalized.append(" ")
            source_starts.append(position)
            source_ends.append(end)
            position = end
        else:
            normalized.append(text[position])
            source_starts.append(position)
            source_ends.append(position + 1)
            position += 1
    return "".join(normalized), source_starts, source_ends

def _collapse_whitespace(text):
    return _collapse_whitespace_with_map(text)[0]

def align_flair_records(paragraph, raw_output):
    """אותו יישור רציף מהמחברת: משפט Flair -> מקור מנורמל -> מקור מקורי."""
    paragraph_text = paragraph["text"]
    normalized_source, source_starts, source_ends = (
        _collapse_whitespace_with_map(paragraph_text)
    )
    mentions, cursor = [], 0

    for sentence_number, record in enumerate(raw_output):
        sentence_text = record["sentence_text"]
        normalized_sentence = _collapse_whitespace(sentence_text)
        sentence_start = normalized_source.find(normalized_sentence, cursor)

        if not normalized_sentence or sentence_start < 0:
            for entity_number, entity in enumerate(record["entities"]):
                mentions.append(make_mention(
                    paragraph, entity["label"], entity["text"], None, None,
                    entity["score"],
                    f"results[].raw_output[{sentence_number}].entities[{entity_number}]",
                    {"flair_text": entity["text"],
                     "error": f"sentence {sentence_number} cannot be aligned"},
                ))
            continue

        cursor = sentence_start + len(normalized_sentence)
        for entity_number, entity in enumerate(record["entities"]):
            local_start = entity["start_pos"]
            local_end = entity["end_pos"]
            reference = (
                f"results[].raw_output[{sentence_number}].entities[{entity_number}]"
            )

            if not 0 <= local_start < local_end <= len(sentence_text):
                mentions.append(make_mention(
                    paragraph, entity["label"], entity["text"], None, None,
                    entity["score"], reference,
                    {"flair_text": entity["text"],
                     "error": "invalid Flair sentence-local offsets"},
                ))
                continue

            normalized_start = (
                sentence_start
                + len(_collapse_whitespace(sentence_text[:local_start]))
            )
            normalized_end = (
                sentence_start
                + len(_collapse_whitespace(sentence_text[:local_end]))
            )

            if not 0 <= normalized_start < normalized_end <= len(normalized_source):
                mentions.append(make_mention(
                    paragraph, entity["label"], entity["text"], None, None,
                    entity["score"], reference,
                    {"flair_text": entity["text"],
                     "error": "mapped offsets outside source"},
                ))
                continue

            start = source_starts[normalized_start]
            end = source_ends[normalized_end - 1]
            original_text = paragraph_text[start:end]

            if _collapse_whitespace(original_text) != _collapse_whitespace(entity["text"]):
                mentions.append(make_mention(
                    paragraph, entity["label"], entity["text"], None, None,
                    entity["score"], reference,
                    {"flair_text": entity["text"],
                     "error": "Flair entity text differs from aligned source"},
                ))
                continue

            mentions.append(make_mention(
                paragraph, entity["label"], original_text, start, end,
                entity["score"], reference,
                {"flair_text": entity["text"]},
            ))
    return mentions

def _test_flair_alignment():
    cases = [
        ("Paris. Paris.",
         [{"sentence_text": "Paris.", "entities": [
             {"text": "Paris", "start_pos": 0, "end_pos": 5,
              "label": "GPE", "score": 1.0}]},
          {"sentence_text": "Paris.", "entities": [
             {"text": "Paris", "start_pos": 0, "end_pos": 5,
              "label": "GPE", "score": 1.0}]}],
         [(0, 5), (7, 12)]),
        ("Leyte\nLuzon\nKorean War",
         [{"sentence_text": "Leyte Luzon Korean War", "entities": [
             {"text": "Leyte Luzon", "start_pos": 0, "end_pos": 11,
              "label": "EVENT", "score": 1.0}]}],
         [(0, 11)]),
        ("Émile\tZola. Émile\tZola.",
         [{"sentence_text": "Émile Zola.", "entities": [
             {"text": "Émile Zola", "start_pos": 0, "end_pos": 10,
              "label": "PERSON", "score": 1.0}]},
          {"sentence_text": "Émile Zola.", "entities": [
             {"text": "Émile Zola", "start_pos": 0, "end_pos": 10,
              "label": "PERSON", "score": 1.0}]}],
         [(0, 10), (12, 22)]),
    ]
    for text, records, expected in cases:
        paragraph = {"text": text, "start": 100}
        found = align_flair_records(paragraph, records)
        assert [(m["start"], m["end"]) for m in found] == expected
        assert all(m["status_validation"] == "valid" for m in found)
        assert all(text[m["start"]:m["end"]] == m["text"] for m in found)

    failed = align_flair_records(
        {"text": "Paris", "start": 0},
        [{"sentence_text": "London", "entities": [
            {"text": "London", "start_pos": 0, "end_pos": 6,
             "label": "GPE", "score": 1.0}]}],
    )
    assert len(failed) == 1
    assert failed[0]["status_validation"] == "invalid"
    assert failed[0]["error"]
    print("בדיקות Flair עברו: שמות ומשפטים חוזרים, שורות/רווחים, Unicode וכשל התאמה")

_test_flair_alignment()

בדיקות Flair עברו: שמות ומשפטים חוזרים, שורות/רווחים, Unicode וכשל התאמה


תא 13 — Flair OntoNotes (Python)

In [14]:
flair_revision = model_info(FLAIR_ID).sha
flair_model_file = hf_hub_download(
    repo_id=FLAIR_ID,
    filename="pytorch_model.bin",
    revision=flair_revision,
)
flair_ner = SequenceTagger.load(flair_model_file)
flair_splitter = SegtokSentenceSplitter()
flair_labels = sorted(str(item) for item in flair_ner.label_dictionary.get_items())

def extract_flair(paragraph):
    sentences = flair_splitter.split(paragraph["text"])
    if sentences:
        flair_ner.predict(sentences, mini_batch_size=1)

    # קודם שומרים את פלט Flair; רק אחר כך מיישרים היסטים.
    raw_output = []
    for sentence in sentences:
        entities = []
        for span in sentence.get_spans("ner"):
            label = span.get_label("ner")
            entities.append({
                "text": span.text,
                "start_pos": span.start_position,
                "end_pos": span.end_position,
                "label": label.value,
                "score": float(label.score),
            })
        raw_output.append({
            "sentence_text": sentence.text,
            "sentence_start_flair": sentence.start_position,
            "entities": entities,
            "flair_output": sentence.to_dict(),
        })

    mentions = align_flair_records(paragraph, raw_output)
    return mentions, raw_output

flair_output = run_model(
    "flair", FLAIR_ID, flair_revision, None,
    flair_labels,
    {"sentence_splitter": "SegtokSentenceSplitter",
     "alignment_method": "sequential_whitespace_equivalence",
     "threshold": None},
    extract_flair,
)
del flair_ner, flair_splitter
release_model()

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2026-10-11 19:16:37,683 SequenceTagger predicts: Dictionary with 76 tags: <unk>, O, B-CARDINAL, E-CARDINAL, S-PERSON, S-CARDINAL, S-PRODUCT, B-PRODUCT, I-PRODUCT, E-PRODUCT, B-WORK_OF_ART, I-WORK_OF_ART, E-WORK_OF_ART, B-PERSON, E-PERSON, S-GPE, B-DATE, I-DATE, E-DATE, S-ORDINAL, S-LANGUAGE, I-PERSON, S-EVENT, S-DATE, B-QUANTITY, E-QUANTITY, S-TIME, B-TIME, I-TIME, E-TIME, B-GPE, E-GPE, S-ORG, I-GPE, S-NORP, B-FAC, I-FAC, E-FAC, B-NORP, E-NORP, S-PERCENT, B-ORG, E-ORG, B-LANGUAGE, E-LANGUAGE, I-CARDINAL, I-ORG, S-WORK_OF_ART, I-QUANTITY, B-MONEY
flair: 10/300
flair: 20/300
flair: 30/300
flair: 40/300
flair: 50/300
flair: 60/300
flair: 70/300
flair: 80/300
flair: 90/300
flair: 100/300
flair: 110/300
flair: 120/300
flair: 130/300
flair: 140/300
flair: 150/300
flair: 160/300
flair: 170/300
flair: 180/300
flair: 190/300
flair: 200/300
flair: 210/300
flair: 220/300
flair: 230/300
flair: 240/300
flair: 250/300
flair: 260/300
flair: 270/300
flair: 280/300
flair: 290/300
flair: 300/300
flair {

תא 14 — שילוב PERSON/GEO/ORG_GROUP ו־DATE/TIME בנפרד (Python)

In [15]:
CORE_MODELS = ("bert", "spacy", "gliner")
EVALUATION_GROUPS = ("PERSON", "GEO", "ORG_GROUP")

GROUP_MAP = {
    "bert": {
        "PER": "PERSON",
        "LOC": "GEO",
        "ORG": "ORG_GROUP",
    },
    "spacy": {
        "PERSON": "PERSON",
        "GPE": "GEO",
        "LOC": "GEO",
        "FAC": "GEO",
        "ORG": "ORG_GROUP",
        "NORP": "NATIONALITY_GROUP",
    },
    "gliner": {
        "person": "PERSON",
        "location": "GEO",
        "geopolitical entity": "GEO",
        "facility": "GEO",
        "organization": "ORG_GROUP",
        "military unit": "ORG_GROUP",
        "nationality or political group": "NATIONALITY_GROUP",
    },
}
assert GROUP_MAP["spacy"]["NORP"] != "ORG_GROUP"
assert (
    GROUP_MAP["gliner"]["nationality or political group"]
    != "ORG_GROUP"
)

TEMPORAL_LABELS = {
    "spacy": {"DATE", "TIME"},
    "flair": {"DATE", "TIME"},
}

outputs = {
    "bert": bert_output,
    "roberta": roberta_output,
    "spacy": spacy_output,
    "gliner": gliner_output,
    "flair": flair_output,
}

candidates = []
temporal_candidates = []

for sample_index, paragraph in enumerate(sample):
    grouped = defaultdict(list)

    for model_name in CORE_MODELS:
        result = outputs[model_name]["results"][sample_index]
        for ordinal, mention in enumerate(result["mentions"]):
            if mention["status_validation"] != "valid":
                continue

            group = GROUP_MAP[model_name].get(mention["label"])
            if group is None:
                continue

            # אותו span ואותה קבוצה בלבד; גבולות שונים נשארים נפרדים.
            key = (group, mention["start"], mention["end"])
            grouped[key].append({
                "model": model_name,
                "label_original": mention["label"],
                "score_original": mention["score"],
                "mention_ordinal": ordinal,
                "ref_output_raw": mention["ref_output_raw"],
            })

    for (group, start, end), support in grouped.items():
        supporting_models = sorted({item["model"] for item in support})

        if group not in EVALUATION_GROUPS:
            decision = "pending_separate_group"
        elif len(supporting_models) >= 2:
            decision = "accepted_experimental"
        elif (
            len(supporting_models) == 1
            and any(
                item["score_original"] is not None
                and item["score_original"] >= HIGH_SINGLETON_SCORE
                for item in support
            )
        ):
            decision = "accepted_experimental_high_singleton"
        else:
            decision = "pending"

        candidates.append({
            "sample_index": sample_index,
            "article_title": paragraph["article_title"],
            "paragraph_index": paragraph["paragraph_index"],
            "text": paragraph["text"][start:end],
            "start": start,
            "end": end,
            "start_document": paragraph["start"] + start,
            "end_document": paragraph["start"] + end,
            "group_mapped": group,
            "supporting_models": supporting_models,
            "support": support,
            "labels_original": [
                item["label_original"] for item in support
            ],
            "scores_original": [
                item["score_original"] for item in support
            ],
            "decision_status": decision,
            "boundary_conflict": False,
        })

    for model_name in TEMPORAL_LABELS:
        result = outputs[model_name]["results"][sample_index]
        for ordinal, mention in enumerate(result["mentions"]):
            if (
                mention["status_validation"] == "valid"
                and mention["label"] in TEMPORAL_LABELS[model_name]
            ):
                temporal_candidates.append({
                    "sample_index": sample_index,
                    "text": mention["text"],
                    "start": mention["start"],
                    "end": mention["end"],
                    "start_document": mention["start_document"],
                    "end_document": mention["end_document"],
                    "label_original": mention["label"],
                    "source_model": model_name,
                    "score_original": mention["score"],
                    "mention_ordinal": ordinal,
                    "ref_output_raw": mention["ref_output_raw"],
                    "temporal_role": "unassigned",
                })

# חפיפה עם גבולות שונים מסומנת על שני המועמדים, בלי לאחד אותם.
by_paragraph_group = defaultdict(list)
for candidate in candidates:
    by_paragraph_group[
        (candidate["sample_index"], candidate["group_mapped"])
    ].append(candidate)

for items in by_paragraph_group.values():
    for left_index, left in enumerate(items):
        for right in items[left_index + 1:]:
            overlaps = (
                max(left["start"], right["start"])
                < min(left["end"], right["end"])
            )
            different_boundaries = (
                (left["start"], left["end"])
                != (right["start"], right["end"])
            )
            if overlaps and different_boundaries:
                left["boundary_conflict"] = True
                right["boundary_conflict"] = True

candidates.sort(
    key=lambda item: (
        item["sample_index"],
        item["start"],
        item["end"],
        item["group_mapped"],
    )
)
temporal_candidates.sort(
    key=lambda item: (
        item["sample_index"],
        item["start"],
        item["end"],
        item["source_model"],
    )
)

accepted_statuses = {
    "accepted_experimental",
    "accepted_experimental_high_singleton",
}
selected_mentions = [
    candidate for candidate in candidates
    if candidate["decision_status"] in accepted_statuses
]

combination = {
    "schema_version": "ner-experiment-2",
    "run_id": RUN_ID,
    "sample_file": sample_path.name,
    "sample_sha256": sample_sha256,
    "status": "experimental_evaluation_view",
    "core_models": list(CORE_MODELS),
    "evaluation_groups": list(EVALUATION_GROUPS),
    "exact_span_support_required": 2,
    "high_singleton_raw_score": HIGH_SINGLETON_SCORE,
    "score_note": (
        "Raw model scores are not calibrated accuracy probabilities "
        "and are not directly comparable across models."
    ),
    "group_mapping": GROUP_MAP,
    "candidates": candidates,
    "selected_mentions": selected_mentions,
    "date_time_candidates": temporal_candidates,
    "note": (
        "DATE/TIME are source-text mentions; event time is not inferred. "
        "NATIONALITY_GROUP is outside the core acceptance rule."
    ),
}
combination_path = (
    RUN_DIR / f"combination_experimental_{uuid.uuid4().hex[:8]}.json"
)
save_json(combination_path, combination)

print("כל המועמדים הממופים:", len(candidates))
print("אזכורים שנבחרו בפועל:", len(selected_mentions))
print(
    "הסכמת שני מודלים לפחות:",
    sum(
        item["decision_status"] == "accepted_experimental"
        for item in selected_mentions
    ),
)
print(
    "מודל יחיד עם ציון גולמי גבוה:",
    sum(
        item["decision_status"]
        == "accepted_experimental_high_singleton"
        for item in selected_mentions
    ),
)
print("מועמדי NATIONALITY_GROUP:", sum(
    item["group_mapped"] == "NATIONALITY_GROUP"
    for item in candidates
))
print("מועמדי DATE/TIME:", len(temporal_candidates))
print("קובץ שילוב:", combination_path)
# CORE_MODELS = ("bert", "spacy", "gliner")
# GROUP_MAP = {
#     "bert": {
#         "PER": "PERSON", "LOC": "GEO", "ORG": "ORG_GROUP",
#     },
#     "spacy": {
#         "PERSON": "PERSON",
#         "GPE": "GEO", "LOC": "GEO", "FAC": "GEO",
#         "ORG": "ORG_GROUP", "NORP": "ORG_GROUP",
#     },
#     "gliner": {
#         "person": "PERSON",
#         "location": "GEO", "geopolitical entity": "GEO",
#         "facility": "GEO",
#         "organization": "ORG_GROUP", "military unit": "ORG_GROUP",
#         "nationality or political group": "ORG_GROUP",
#     },
# }
# TEMPORAL_LABELS = {
#     "spacy": {"DATE", "TIME"},
#     "flair": {"DATE", "TIME"},
# }
# outputs = {
#     "bert": bert_output,
#     "roberta": roberta_output,
#     "spacy": spacy_output,
#     "gliner": gliner_output,
#     "flair": flair_output,
# }

# candidates = []
# temporal_candidates = []

# for sample_index, paragraph in enumerate(sample):
#     grouped = defaultdict(list)
#     for model_name in CORE_MODELS:
#         result = outputs[model_name]["results"][sample_index]
#         for ordinal, mention in enumerate(result["mentions"]):
#             if mention["status_validation"] != "valid":
#                 continue
#             group = GROUP_MAP[model_name].get(mention["label"])
#             if group is None:
#                 continue
#             key = (group, mention["start"], mention["end"])
#             grouped[key].append({
#                 "model": model_name,
#                 "label_original": mention["label"],
#                 "score_original": mention["score"],
#                 "mention_ordinal": ordinal,
#                 "ref_output_raw": mention["ref_output_raw"],
#             })

#     for (group, start, end), support in grouped.items():
#         supporting_models = sorted({item["model"] for item in support})
#         if len(supporting_models) >= 2:
#             decision = "accepted_experimental"
#         elif any(
#             item["score_original"] is not None
#             and item["score_original"] >= HIGH_SINGLETON_SCORE
#             for item in support
#         ):
#             decision = "pending_high_singleton"
#         else:
#             decision = "pending"

#         candidates.append({
#             "sample_index": sample_index,
#             "article_title": paragraph["article_title"],
#             "paragraph_index": paragraph["paragraph_index"],
#             "text": paragraph["text"][start:end],
#             "start": start,
#             "end": end,
#             "start_document": paragraph["start"] + start,
#             "end_document": paragraph["start"] + end,
#             "group_mapped": group,
#             "supporting_models": supporting_models,
#             "support": support,
#             "labels_original": [item["label_original"] for item in support],
#             "scores_original": [item["score_original"] for item in support],
#             "decision_status": decision,
#             "boundary_conflict": False,
#         })

#     for model_name in TEMPORAL_LABELS:
#         result = outputs[model_name]["results"][sample_index]
#         for ordinal, mention in enumerate(result["mentions"]):
#             if (mention["status_validation"] == "valid"
#                     and mention["label"] in TEMPORAL_LABELS[model_name]):
#                 temporal_candidates.append({
#                     "sample_index": sample_index,
#                     "text": mention["text"],
#                     "start": mention["start"],
#                     "end": mention["end"],
#                     "start_document": mention["start_document"],
#                     "end_document": mention["end_document"],
#                     "label_original": mention["label"],
#                     "source_model": model_name,
#                     "score_original": mention["score"],
#                     "ref_output_raw": mention["ref_output_raw"],
#                     "temporal_role": "unassigned",
#                 })

# # גבולות שונים נשארים מועמדים שונים, גם אם האזכורים חופפים.
# by_paragraph_group = defaultdict(list)
# for item in candidates:
#     by_paragraph_group[(item["sample_index"], item["group_mapped"])].append(item)
# for items in by_paragraph_group.values():
#     for left_index, left in enumerate(items):
#         for right in items[left_index + 1:]:
#             overlaps = max(left["start"], right["start"]) < min(left["end"], right["end"])
#             different_boundaries = (left["start"], left["end"]) != (right["start"], right["end"])
#             if overlaps and different_boundaries:
#                 left["boundary_conflict"] = True
#                 right["boundary_conflict"] = True

# candidates.sort(key=lambda x: (
#     x["sample_index"], x["start"], x["end"], x["group_mapped"]
# ))
# temporal_candidates.sort(key=lambda x: (
#     x["sample_index"], x["start"], x["end"], x["source_model"]
# ))

# combination = {
#     "schema_version": "ner-experiment-1",
#     "run_id": RUN_ID,
#     "sample_file": sample_path.name,
#     "sample_sha256": sample_sha256,
#     "status": "experimental_evaluation_view",
#     "core_models": list(CORE_MODELS),
#     "exact_span_support_required": 2,
#     "high_singleton_raw_score": HIGH_SINGLETON_SCORE,
#     "score_note": "Raw model scores are not directly comparable or calibrated.",
#     "candidates": candidates,
#     "date_time_candidates": temporal_candidates,
#     "note": "DATE/TIME are text mentions; event time is not inferred.",
# }
# combination_path = RUN_DIR / "combination_experimental.json"
# save_json(combination_path, combination)

# print("מועמדי ליבה:", len(candidates))
# print("accepted_experimental:", sum(
#     c["decision_status"] == "accepted_experimental" for c in candidates
# ))
# print("pending_high_singleton:", sum(
#     c["decision_status"] == "pending_high_singleton" for c in candidates
# ))
# print("boundary_conflict:", sum(c["boundary_conflict"] for c in candidates))
# print("מועמדי DATE/TIME:", len(temporal_candidates))

כל המועמדים הממופים: 11999
אזכורים שנבחרו בפועל: 5455
הסכמת שני מודלים לפחות: 5090
מודל יחיד עם ציון גולמי גבוה: 365
מועמדי NATIONALITY_GROUP: 2134
מועמדי DATE/TIME: 2934
קובץ שילוב: /content/TMP_NER/20261011T190026Z_ce33d695/combination_experimental_b4aa4c5d.json


תא 15 — קריאה חוזרת, בדיקות, manifest ו־ZIP נקי (Python)

In [16]:
saved_sample = json.loads(sample_path.read_text(encoding="utf-8"))
assert saved_sample["run_id"] == RUN_ID
assert saved_sample["source_sha256"] == source_sha256
assert saved_sample["sample_size"] == SAMPLE_SIZE
assert len(saved_sample["paragraphs"]) == SAMPLE_SIZE
assert sha256_file(sample_path) == sample_sha256
validate_sample(saved_sample["paragraphs"], data)

model_summary = {}
saved_models = {}

for short_name, path in MODEL_FILES.items():
    saved = json.loads(path.read_text(encoding="utf-8"))
    saved_models[short_name] = saved

    assert saved["run_id"] == RUN_ID
    assert saved["sample_sha256"] == sample_sha256
    assert saved["sample_size"] == SAMPLE_SIZE
    assert len(saved["results"]) == SAMPLE_SIZE

    counts = {
        "success": 0,
        "partial_invalid": 0,
        "error": 0,
        "valid_mentions": 0,
        "invalid_mentions": 0,
        "zero_mentions": 0,
    }

    for index, result in enumerate(saved["results"]):
        assert result["sample_index"] == index
        assert result["paragraph"] == saved_sample["paragraphs"][index]

        status = result["status_processing"]
        assert status in ("success", "partial_invalid", "error")
        counts[status] += 1

        assert isinstance(result["mentions"], list)
        if status == "success" and not result["mentions"]:
            counts["zero_mentions"] += 1
        if status != "success":
            assert result["error"]

        for mention in result["mentions"]:
            assert mention["status_validation"] in (
                "valid", "pending", "invalid"
            )
            if mention["status_validation"] == "valid":
                check_mention(result["paragraph"], mention)
                counts["valid_mentions"] += 1
            elif mention["status_validation"] == "invalid":
                assert mention["error"]
                counts["invalid_mentions"] += 1

    model_summary[short_name] = counts

assert set(saved_models) == {
    "bert", "roberta", "spacy", "gliner", "flair"
}

saved_combination = json.loads(
    combination_path.read_text(encoding="utf-8")
)
assert saved_combination["run_id"] == RUN_ID
assert saved_combination["sample_sha256"] == sample_sha256
assert saved_combination["schema_version"] == "ner-experiment-2"

saved_candidates = saved_combination["candidates"]
saved_selected = saved_combination["selected_mentions"]

def candidate_key(candidate):
    return (
        candidate["sample_index"],
        candidate["group_mapped"],
        candidate["start"],
        candidate["end"],
    )

candidate_by_key = {}
for candidate in saved_candidates:
    key = candidate_key(candidate)
    assert key not in candidate_by_key, f"כפילות span וקבוצה: {key}"
    candidate_by_key[key] = candidate

    index = candidate["sample_index"]
    assert 0 <= index < SAMPLE_SIZE
    paragraph = saved_sample["paragraphs"][index]
    start, end = candidate["start"], candidate["end"]

    assert 0 <= start < end <= len(paragraph["text"])
    assert paragraph["text"][start:end] == candidate["text"]
    assert candidate["start_document"] == paragraph["start"] + start
    assert candidate["end_document"] == paragraph["start"] + end
    assert data[paragraph["article_title"]][
        candidate["start_document"]:candidate["end_document"]
    ] == candidate["text"]

    models = sorted({item["model"] for item in candidate["support"]})
    assert candidate["supporting_models"] == models
    assert candidate["labels_original"] == [
        item["label_original"] for item in candidate["support"]
    ]
    assert candidate["scores_original"] == [
        item["score_original"] for item in candidate["support"]
    ]

    # כל רשומת תמיכה נבדקת מול אזכור המקור שנשמר בקובץ המודל.
    for support in candidate["support"]:
        model_name = support["model"]
        ordinal = support["mention_ordinal"]
        assert model_name in CORE_MODELS

        original = saved_models[model_name]["results"][index]["mentions"][ordinal]
        assert original["status_validation"] == "valid"
        assert original["start"] == start
        assert original["end"] == end
        assert original["text"] == candidate["text"]
        assert original["label"] == support["label_original"]
        assert original["score"] == support["score_original"]
        assert original["ref_output_raw"] == support["ref_output_raw"]
        assert GROUP_MAP[model_name][original["label"]] == candidate["group_mapped"]

    if candidate["group_mapped"] not in EVALUATION_GROUPS:
        expected_status = "pending_separate_group"
    elif len(models) >= 2:
        expected_status = "accepted_experimental"
    elif (
        len(models) == 1
        and any(
            item["score_original"] is not None
            and item["score_original"] >= HIGH_SINGLETON_SCORE
            for item in candidate["support"]
        )
    ):
        expected_status = "accepted_experimental_high_singleton"
    else:
        expected_status = "pending"

    assert candidate["decision_status"] == expected_status

# NATIONALITY_GROUP אינה נספרת כ־ORG_GROUP בהערכה.
assert GROUP_MAP["spacy"]["NORP"] == "NATIONALITY_GROUP"
assert (
    GROUP_MAP["gliner"]["nationality or political group"]
    == "NATIONALITY_GROUP"
)
assert all(
    item["group_mapped"] in EVALUATION_GROUPS
    for item in saved_selected
)

selected_keys = [candidate_key(item) for item in saved_selected]
assert len(selected_keys) == len(set(selected_keys))
assert set(selected_keys) == {
    key for key, item in candidate_by_key.items()
    if item["decision_status"] in {
        "accepted_experimental",
        "accepted_experimental_high_singleton",
    }
}
for selected in saved_selected:
    assert selected == candidate_by_key[candidate_key(selected)]

for temporal in saved_combination["date_time_candidates"]:
    index = temporal["sample_index"]
    paragraph = saved_sample["paragraphs"][index]
    start, end = temporal["start"], temporal["end"]
    assert 0 <= start < end <= len(paragraph["text"])
    assert paragraph["text"][start:end] == temporal["text"]
    assert temporal["start_document"] == paragraph["start"] + start
    assert temporal["end_document"] == paragraph["start"] + end
    assert temporal["temporal_role"] == "unassigned"
    assert temporal["source_model"] in ("spacy", "flair")

run_success = all(
    counts["error"] == 0 and counts["partial_invalid"] == 0
    for counts in model_summary.values()
)

summary = {
    "schema_version": "ner-experiment-2",
    "run_id": RUN_ID,
    "sample_size": SAMPLE_SIZE,
    "sample_sha256": sample_sha256,
    "models": model_summary,
    "run_status": (
        "success" if run_success else "completed_with_errors"
    ),
    "candidate_count": len(saved_candidates),
    "selected_count": len(saved_selected),
    "overlap_with_previous_samples": overlap_report,
    "checks": (
        "Saved files reloaded; hashes, counts, source/document offsets, "
        "model support, decisions and unique selected spans verified."
    ),
}
summary_path = (
    RUN_DIR / f"validation_summary_{uuid.uuid4().hex[:8]}.json"
)
save_json(summary_path, summary)

# רשימה מפורשת בלבד; אין כיווץ גורף של /content/TMP_NER.
zip_members = {
    f"TMP_NER/{sample_path.name}": sample_path,
    "TMP_NER/ner_bert.json": MODEL_FILES["bert"],
    "TMP_NER/ner_roberta.json": MODEL_FILES["roberta"],
    "TMP_NER/ner_spacy.json": MODEL_FILES["spacy"],
    "TMP_NER/ner_gliner.json": MODEL_FILES["gliner"],
    "TMP_NER/ner_flair.json": MODEL_FILES["flair"],
    "combination_experimental.json": combination_path,
    "validation_summary.json": summary_path,
}

manifest = {
    "schema_version": "ner-experiment-2",
    "run_id": RUN_ID,
    "run_status": summary["run_status"],
    "sample_file": f"TMP_NER/{sample_path.name}",
    "sample_sha256": sample_sha256,
    "source_file": input_path.name,
    "source_sha256": source_sha256,
    "seed": SEED,
    "sample_size": SAMPLE_SIZE,
    "files_sha256": {
        archive_name: sha256_file(path)
        for archive_name, path in zip_members.items()
    },
}
manifest_path = RUN_DIR / f"manifest_{uuid.uuid4().hex[:8]}.json"
save_json(manifest_path, manifest)
zip_members["manifest.json"] = manifest_path

# גם הרצה חוזרת של התא באותו runtime לא תדרוס ZIP קודם.
if ZIP_PATH.exists():
    ZIP_PATH = ZIP_PATH.with_name(
        f"{ZIP_PATH.stem}_{uuid.uuid4().hex[:8]}.zip"
    )

with zipfile.ZipFile(
    ZIP_PATH, "x", compression=zipfile.ZIP_DEFLATED
) as archive:
    for archive_name, path in zip_members.items():
        archive.write(path, arcname=archive_name)

with zipfile.ZipFile(ZIP_PATH) as archive:
    actual_names = archive.namelist()
    assert actual_names == list(zip_members)

    for archive_name, expected_hash in manifest["files_sha256"].items():
        assert sha256_bytes(archive.read(archive_name)) == expected_hash

    reloaded_sample = json.loads(
        archive.read(f"TMP_NER/{sample_path.name}")
    )
    assert len(reloaded_sample["paragraphs"]) == SAMPLE_SIZE
    assert sha256_bytes(
        archive.read(f"TMP_NER/{sample_path.name}")
    ) == sample_sha256

    reloaded_combination = json.loads(
        archive.read("combination_experimental.json")
    )
    assert reloaded_combination["selected_mentions"] == saved_selected
    assert len(reloaded_combination["selected_mentions"]) == len(selected_keys)

print("מצב הרצה:", summary["run_status"])
print("קובצי ZIP, בדיוק:")
for archive_name in actual_names:
    print(" -", archive_name)
print("ZIP:", ZIP_PATH)
files.download(str(ZIP_PATH))


# saved_sample = json.loads(sample_path.read_text(encoding="utf-8"))
# assert saved_sample["run_id"] == RUN_ID
# assert saved_sample["source_sha256"] == source_sha256
# assert saved_sample["sample_size"] == SAMPLE_SIZE
# assert len(saved_sample["paragraphs"]) == SAMPLE_SIZE
# assert sha256_file(sample_path) == sample_sha256
# validate_sample(saved_sample["paragraphs"], data)

# model_summary = {}
# for short_name, path in MODEL_FILES.items():
#     saved = json.loads(path.read_text(encoding="utf-8"))
#     assert saved["run_id"] == RUN_ID
#     assert saved["sample_sha256"] == sample_sha256
#     assert saved["sample_size"] == SAMPLE_SIZE
#     assert len(saved["results"]) == SAMPLE_SIZE

#     counts = {"success": 0, "partial_invalid": 0, "error": 0,
#               "valid_mentions": 0, "invalid_mentions": 0, "zero_mentions": 0}
#     for index, result in enumerate(saved["results"]):
#         assert result["sample_index"] == index
#         assert result["paragraph"] == saved_sample["paragraphs"][index]
#         status = result["status_processing"]
#         assert status in ("success", "partial_invalid", "error")
#         counts[status] += 1
#         assert isinstance(result["mentions"], list)
#         if status == "success" and not result["mentions"]:
#             counts["zero_mentions"] += 1
#         if status != "success":
#             assert result["error"]
#         for mention in result["mentions"]:
#             assert mention["status_validation"] in ("valid", "pending", "invalid")
#             if mention["status_validation"] == "valid":
#                 check_mention(result["paragraph"], mention)
#                 counts["valid_mentions"] += 1
#             elif mention["status_validation"] == "invalid":
#                 assert mention["error"]
#                 counts["invalid_mentions"] += 1
#     model_summary[short_name] = counts

# saved_combination = json.loads(
#     combination_path.read_text(encoding="utf-8")
# )
# assert saved_combination["sample_sha256"] == sample_sha256
# for candidate in saved_combination["candidates"]:
#     paragraph = saved_sample["paragraphs"][candidate["sample_index"]]
#     assert paragraph["text"][candidate["start"]:candidate["end"]] == candidate["text"]
#     assert candidate["start_document"] == paragraph["start"] + candidate["start"]
#     assert candidate["end_document"] == paragraph["start"] + candidate["end"]
#     assert candidate["supporting_models"] == sorted(
#         {item["model"] for item in candidate["support"]}
#     )
#     if candidate["decision_status"] == "accepted_experimental":
#         assert len(candidate["supporting_models"]) >= 2

# for candidate in saved_combination["date_time_candidates"]:
#     paragraph = saved_sample["paragraphs"][candidate["sample_index"]]
#     assert paragraph["text"][candidate["start"]:candidate["end"]] == candidate["text"]
#     assert candidate["temporal_role"] == "unassigned"

# run_success = all(
#     counts["error"] == 0 and counts["partial_invalid"] == 0
#     for counts in model_summary.values()
# )
# summary = {
#     "schema_version": "ner-experiment-1",
#     "run_id": RUN_ID,
#     "sample_size": SAMPLE_SIZE,
#     "sample_sha256": sample_sha256,
#     "models": model_summary,
#     "run_status": "success" if run_success else "completed_with_errors",
#     "overlap_with_previous_samples": overlap_report,
#     "checks": "saved JSON reloaded; hashes, counts, paragraph/document offsets verified",
# }
# summary_path = RUN_DIR / "validation_summary.json"
# save_json(summary_path, summary)

# # רשימה מפורשת בלבד: שום קובץ ישן מתוך /content/TMP_NER לא ייכנס.
# zip_members = {
#     f"TMP_NER/{sample_path.name}": sample_path,
#     "TMP_NER/ner_bert.json": MODEL_FILES["bert"],
#     "TMP_NER/ner_roberta.json": MODEL_FILES["roberta"],
#     "TMP_NER/ner_spacy.json": MODEL_FILES["spacy"],
#     "TMP_NER/ner_gliner.json": MODEL_FILES["gliner"],
#     "TMP_NER/ner_flair.json": MODEL_FILES["flair"],
#     "combination_experimental.json": combination_path,
#     "validation_summary.json": summary_path,
# }
# manifest = {
#     "schema_version": "ner-experiment-1",
#     "run_id": RUN_ID,
#     "run_status": summary["run_status"],
#     "sample_file": f"TMP_NER/{sample_path.name}",
#     "sample_sha256": sample_sha256,
#     "source_file": input_path.name,
#     "source_sha256": source_sha256,
#     "seed": SEED,
#     "sample_size": SAMPLE_SIZE,
#     "files_sha256": {
#         archive_name: sha256_file(path)
#         for archive_name, path in zip_members.items()
#     },
# }
# manifest_path = RUN_DIR / "manifest.json"
# save_json(manifest_path, manifest)
# zip_members["manifest.json"] = manifest_path

# with zipfile.ZipFile(ZIP_PATH, "x", compression=zipfile.ZIP_DEFLATED) as archive:
#     for archive_name, path in zip_members.items():
#         archive.write(path, arcname=archive_name)

# with zipfile.ZipFile(ZIP_PATH) as archive:
#     actual_names = archive.namelist()
#     assert actual_names == list(zip_members)
#     for archive_name, expected_hash in manifest["files_sha256"].items():
#         assert sha256_bytes(archive.read(archive_name)) == expected_hash
#     reloaded_sample = json.loads(
#         archive.read(f"TMP_NER/{sample_path.name}")
#     )
#     assert len(reloaded_sample["paragraphs"]) == SAMPLE_SIZE
#     assert sha256_bytes(
#         archive.read(f"TMP_NER/{sample_path.name}")
#     ) == sample_sha256

# print("מצב הרצה:", summary["run_status"])
# print("ספירות לפי מודל:")
# for name, counts in model_summary.items():
#     print(name, counts)
# print("קובצי ZIP, בדיוק:")
# for archive_name in actual_names:
#     print(" -", archive_name)
# print("ZIP:", ZIP_PATH)
# files.download(str(ZIP_PATH))

מצב הרצה: success
קובצי ZIP, בדיוק:
 - TMP_NER/sample_300_paragraphs_seed17.json
 - TMP_NER/ner_bert.json
 - TMP_NER/ner_roberta.json
 - TMP_NER/ner_spacy.json
 - TMP_NER/ner_gliner.json
 - TMP_NER/ner_flair.json
 - combination_experimental.json
 - validation_summary.json
 - manifest.json
ZIP: /content/ner_run_20261011T190026Z_ce33d695.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

סיכום הבדיקות

In [17]:
with zipfile.ZipFile(ZIP_PATH) as archive:
    exported_combination = json.loads(
        archive.read("combination_experimental.json")
    )
    exported_summary = json.loads(
        archive.read("validation_summary.json")
    )

all_candidates = exported_combination["candidates"]
selected = exported_combination["selected_mentions"]

status_counts = defaultdict(int)
group_counts = defaultdict(int)
selected_group_counts = defaultdict(int)
model_counts = defaultdict(int)

for item in all_candidates:
    status_counts[item["decision_status"]] += 1
    group_counts[item["group_mapped"]] += 1

for item in selected:
    selected_group_counts[item["group_mapped"]] += 1
    for model_name in item["supporting_models"]:
        model_counts[model_name] += 1

print("מצב הרצה:", exported_summary["run_status"])
print("כל המועמדים לפי סטטוס שילוב:")
for key in sorted(status_counts):
    print(f"  {key}: {status_counts[key]}")

print("כל המועמדים לפי קבוצת תוויות:")
for key in sorted(group_counts):
    print(f"  {key}: {group_counts[key]}")

print("אזכורים שנבחרו בפועל לפי קבוצת תוויות:")
for key in sorted(selected_group_counts):
    print(f"  {key}: {selected_group_counts[key]}")

print("אזכורים שנבחרו בפועל לפי מודל תומך:")
for key in sorted(model_counts):
    print(f"  {key}: {model_counts[key]}")

print("פסקאות עם כשל עיבוד או ולידציה לפי מודל:")
for model_name, counts in sorted(exported_summary["models"].items()):
    print(
        f"  {model_name}: error={counts['error']}, "
        f"partial_invalid={counts['partial_invalid']}"
    )

overlap = exported_summary["overlap_with_previous_samples"]
print("בדיקת חפיפה:", overlap["status"])
if overlap["status"] == "checked":
    print("מספר פסקאות חופפות ייחודיות:", overlap["overlap_count"])
    print("מזהי הפסקאות החופפות:")
    for paragraph_id in overlap["overlapping_paragraphs"]:
        print(" ", paragraph_id)
else:
    print("not_checked — לא נמצא קובץ מדגם קודם בר־השוואה")

מצב הרצה: success
כל המועמדים לפי סטטוס שילוב:
  accepted_experimental: 5090
  accepted_experimental_high_singleton: 365
  pending: 4410
  pending_separate_group: 2134
כל המועמדים לפי קבוצת תוויות:
  GEO: 4448
  NATIONALITY_GROUP: 2134
  ORG_GROUP: 3507
  PERSON: 1910
אזכורים שנבחרו בפועל לפי קבוצת תוויות:
  GEO: 2970
  ORG_GROUP: 1241
  PERSON: 1244
אזכורים שנבחרו בפועל לפי מודל תומך:
  bert: 5061
  gliner: 3320
  spacy: 4458
פסקאות עם כשל עיבוד או ולידציה לפי מודל:
  bert: error=0, partial_invalid=0
  flair: error=0, partial_invalid=0
  gliner: error=0, partial_invalid=0
  roberta: error=0, partial_invalid=0
  spacy: error=0, partial_invalid=0
בדיקת חפיפה: checked
מספר פסקאות חופפות ייחודיות: 4
מזהי הפסקאות החופפות:
  {'sample_index': 123, 'article_title': 'British Empire in World War II', 'paragraph_index': 29, 'start': 27060, 'end': 27448}
  {'sample_index': 136, 'article_title': 'Declaration by United Nations', 'paragraph_index': 11, 'start': 7129, 'end': 7378}
  {'sample_index': 